# Indexación RAG Local para Agente de Onboarding

Este notebook lee los archivos en la carpeta `data/` (ya sean archivos de texto `.txt` o documentos `.pdf`), genera fragmentos (chunks), calcula sus embeddings e inicializa una base vectorial local utilizando **FAISS** y **LlamaIndex**.

## Requisitos previos

Asegúrate de haber instalado las dependencias correspondientes:
```bash
uv sync --extra notebooks --extra adk --extra rag
```
Y de haber configurado tu archivo `.env` en la raíz del proyecto.

In [1]:
from pathlib import Path
import os
import sys

def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists():
            return path
    raise RuntimeError("No se pudo encontrar la raíz del repositorio")

def load_env_file(env_file: Path) -> None:
    if not env_file.exists():
        print("No se encontró el archivo .env. Usando variables de entorno del sistema.")
        return
    for line in env_file.read_text().splitlines():
        line = line.strip()
        if "=" in line and not line.startswith("#"):
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")
    print(f"Variables cargadas de {env_file.name}")

REPO_ROOT = find_repo_root(Path.cwd())
ENV_FILE = REPO_ROOT / ".env"
load_env_file(ENV_FILE)

SRC_DIR = REPO_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from onboarding.model_config import get_embedding_model

DATA_DIR = REPO_ROOT / "data"
STORAGE_DIR = REPO_ROOT / "faiss_index"

print(f"Directorio de datos: {DATA_DIR.relative_to(REPO_ROOT)}")
print(f"Directorio del índice: {STORAGE_DIR.relative_to(REPO_ROOT)}")
print(f"Proveedor de modelos: {os.getenv('MODEL_PROVIDER', 'gemini')}")
print(f"Modelo de embeddings: {os.getenv('LOCAL_RAG_EMBEDDING_MODEL', 'gemini-embedding-001')}")

## 1. Cargar documentos y generar fragmentos (chunks)

Usamos `SimpleDirectoryReader` para cargar automáticamente todos los archivos compatibles en el directorio `data/` (como `.txt`, `.md` o `.pdf`). Luego dividimos los textos en fragmentos con un splitter de oraciones para conservar el contexto semántico.

In [2]:
from llama_index.core import SimpleDirectoryReader
from llama_index.core.node_parser import SentenceSplitter

if not DATA_DIR.exists():
    raise FileNotFoundError(f"El directorio de datos {DATA_DIR} no existe.")

print(f"Cargando documentos de: {DATA_DIR}")
documents = SimpleDirectoryReader(input_dir=str(DATA_DIR)).load_data()

splitter = SentenceSplitter(chunk_size=500, chunk_overlap=50)
nodes = splitter.get_nodes_from_documents(documents)

print(f"Cargados {len(documents)} archivos/páginas de documentos.")
print(f"Creados {len(nodes)} fragmentos (chunks).")
if len(nodes) > 0:
    print("\nVista previa del primer fragmento:")
    print(nodes[0].text[:400])

## 2. Crear y persistir el índice local de FAISS

Calculamos los embeddings llamando al modelo correspondiente e indexamos localmente los fragmentos.

In [3]:
import shutil
import faiss
from llama_index.core import StorageContext, VectorStoreIndex
from llama_index.vector_stores.faiss import FaissVectorStore

embed_model = get_embedding_model()
embedding_dim = len(embed_model.get_text_embedding("dimension probe"))

if STORAGE_DIR.exists():
    print(f"Limpiando índice anterior en {STORAGE_DIR}...")
    shutil.rmtree(STORAGE_DIR)

faiss_index = faiss.IndexFlatL2(embedding_dim)
vector_store = FaissVectorStore(faiss_index=faiss_index)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

index = VectorStoreIndex(
    nodes,
    storage_context=storage_context,
    embed_model=embed_model,
)
index.storage_context.persist(persist_dir=str(STORAGE_DIR))

print(f"Dimensión del embedding: {embedding_dim}")
print(f"Índice local FAISS persistido con éxito en {STORAGE_DIR.relative_to(REPO_ROOT)}")

## 3. Cargar y verificar la recuperación de datos

Simulamos lo que hará el agente al arrancar: recargar el índice y realizar una consulta de prueba.

In [4]:
from llama_index.core import load_index_from_storage

vector_store = FaissVectorStore.from_persist_dir(str(STORAGE_DIR))
storage_context = StorageContext.from_defaults(
    vector_store=vector_store,
    persist_dir=str(STORAGE_DIR),
)
loaded_index = load_index_from_storage(
    storage_context=storage_context,
    embed_model=embed_model,
)

retriever = loaded_index.as_retriever(similarity_top_k=2)
query = "Cuántos días de vacaciones tengo y cómo los pido?"
matches = retriever.retrieve(query)

print(f"Consulta de prueba: '{query}'\n")
for i, match in enumerate(matches, start=1):
    print(f"--- Coincidencia {i} | Puntuación={match.score:.4f} ---")
    print(match.node.text.strip())
    print()